# Kolmogorov Navier–Stokes — corrected JCP five-model numerical comparison

This is a **new, independent** experimental suite, following the original-model tables in **Burgers** and **Boussinesq**, **not** a continuation of the superseded Navier–Stokes 22-run sensitivity table.

**Models:** TB-AWFNO (training-data temporal SVD basis, 12 temporal modes), AWFNO (LGL), standard WFNO (K-window), **WFNO-PM** (width chosen to match AWFNO's trainable *real scalar degrees of freedom*, including complex Fourier tensors), and **full-trajectory neuraloperator FNO** (directly maps one initial vorticity field to all subsequent 100 frames; **not a K-window FNO**). Persistence is a no-training reference.

**One shared PDE benchmark**: verified `300×128×128×101`, absolute times `[0.25,0.50]`, periodic square `[0,2π)^2`, Re=500, source `7030bc43e6331cdf472182ae0ed4f2684c18da8d336ca9ed92c37baf0bfc8a06`. Disjoint train/validation/test 200/30/70. Sample sizes `25,50,100,200`. Models use four blocks, width 32 for the proposed architectures, modes 16×16, local radius 2, original LGL ordering; K20 for all windowed models, temporal basis rank12. All methods receive one true initial field at test.

**Metrics, as in Boussinesq:** absolute physical full space–time (L^2) (trapezoidal time quadrature and periodic-area element), final-time spatial (L^2), mean/std/median across all 70 test trajectories; element count versus **real scalar DOF**; training seconds; full-test inference seconds/per trajectory; GPU peak allocated MiB. Outputs include **wide and long tables**, full/final error vs sample-size, per-time error curves, true/predicted/error vorticity comparisons at t=0,0.0625,0.125,0.1875,0.25, and validation training curves. Tables save after each N, and completed N are skipped on restart; in-progress models write epoch checkpoints.

**Key protocol distinction:** windowed methods are autoregressive K20; native FNO directly predicts all 100 steps. Differences in the number of training examples, inference operations and effective optimization exposure must be reported. Training is selected via training-window/full-query validation L2; test results are never used for checkpoint selection.

**This notebook has not yet produced measurements.** First it checks data shape and the actual five architectures (including finite backward gradients). Training and validation follow only after smoke passes.

Source is version-pinned. The prior 22-run ablation folder remains unchanged.

In [ ]:
#@title 1. Mount verified dataset, configure GPU
from google.colab import drive
from pathlib import Path
import torch, os, sys, subprocess, json, time, hashlib
drive.mount('/content/drive')
assert torch.cuda.is_available(),'Runtime > Change runtime type > GPU'
DATA_DIR=Path('/content/drive/MyDrive/Kolmogorov_NS_Re500_start0.25_horizon0.25_dt0.0025_v1')
DATASET=DATA_DIR/'kolmogorov_dataset.pt'
OUTPUT_ROOT=Path('/content/drive/MyDrive/AWFNO_NS_Re500_JCP_FIVE_MODELS_v1')
assert DATASET.is_file(),DATASET
manifest=json.loads((DATA_DIR/'metadata.json').read_text())
assert manifest['status']=='PASS'
assert manifest['shape']==[300,128,128,101]
assert manifest['sha256']=='7030bc43e6331cdf472182ae0ed4f2684c18da8d336ca9ed92c37baf0bfc8a06'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0))
print('Verified data:',DATASET,'|',round(DATASET.stat().st_size/2**30,3),'GiB')
print('Independent output root:',OUTPUT_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','neuraloperator==2.0.0'],
               check=True)
from neuralop.models import FNO
print('Official neuraloperator FNO import: PASS')


In [ ]:
#@title 2. Original five-model study, identical sample sizes to the prior examples
SAMPLE_SIZES='25,50,100,200' #@param {type:'string'}
WINDOW_K=20 #@param {type:'integer'}
WINDOW_STRIDE=20 #@param {type:'integer'}
TB_RANK=12 #@param {type:'integer'}
BATCH_SIZE=8 #@param {type:'integer'}
EPOCHS=120 #@param {type:'integer'}
RUN_ARCHITECTURE_SMOKE=True #@param {type:'boolean'}
START_TRAINING=True #@param {type:'boolean'}
FORCE_RETRAIN=False #@param {type:'boolean'}
assert SAMPLE_SIZES.split(',')==['25','50','100','200'], 'Keep predefined N values.'
assert 1<=TB_RANK<=WINDOW_K<101
assert WINDOW_STRIDE==WINDOW_K, 'Match nonoverlapping Boussinesq-style K-window training.'
assert BATCH_SIZE>=1 and EPOCHS>=1
ENV=os.environ.copy()
ENV.update({
    'NS_DATA_DIR':str(DATA_DIR),
    'NS_OUTPUT_ROOT':str(OUTPUT_ROOT),
    'NS_SIZES':SAMPLE_SIZES,'NS_K':str(WINDOW_K),
    'NS_STRIDE':str(WINDOW_STRIDE),'NS_TB_RANK':str(TB_RANK),
    'NS_BATCH':str(BATCH_SIZE),'NS_EPOCHS':str(EPOCHS),
    'NS_DATA_SHA':manifest['sha256'],
    'PYTHONUNBUFFERED':'1',
})
print('Training N:',SAMPLE_SIZES)
print('Models: TB-AWFNO, AWFNO, WFNO, WFNO-PM, full-trajectory FNO')
print('AWFNO reference: K20, LGL, 4 blocks, 16 modes, radius2')
print('Training budget:',EPOCHS,'epochs per model (early stopping after plateau)')


In [ ]:
#@title 3. Download version-pinned, human-readable corrected model source
from pathlib import Path
import urllib.request,hashlib,py_compile
SOURCE_COMMIT='707e97447befdf8f7974a97fb49733eb980b7dce'
SOURCE_URL=('https://raw.githubusercontent.com/dimigourz/AWFNO/' + SOURCE_COMMIT +
            '/NavierStokes2D/kolmogorov_jcp_model_comparison.py')
DRIVER=Path('/content/kolmogorov_jcp_model_comparison.py')
request=urllib.request.Request(SOURCE_URL,headers={'Cache-Control':'no-cache'})
with urllib.request.urlopen(request,timeout=90) as response:
    data=response.read()
assert len(data)>45000,'Incomplete downloaded code'
DRIVER.write_bytes(data)
py_compile.compile(str(DRIVER),doraise=True)
print('Version-pinned code:',SOURCE_URL)
print('Code SHA256:',hashlib.sha256(data).hexdigest())
print('Standalone code compilation: PASS')


In [ ]:
#@title 4. Five-model smoke test: exact output shapes and backward gradients
if RUN_ARCHITECTURE_SMOKE:
    SMOKE_ENV=ENV.copy();SMOKE_ENV['NS_SMOKE_ONLY']='1'
    logfile=OUTPUT_ROOT/'five_model_smoke.log'
    with logfile.open('w') as log:
        with subprocess.Popen([sys.executable,'-u',str(DRIVER)],
                env=SMOKE_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                bufsize=1,text=True,errors='replace') as proc:
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
            status=proc.wait()
    if status:raise RuntimeError(f'Five-model architecture smoke FAILED; see {logfile}')
    print('[SMOKE TESTS PASSED] Five reference architectures, gradients, output shapes')
else:
    raise RuntimeError('Full study requires architecture smoke tests. Do not bypass.')


In [ ]:
#@title 5. Train ALL five models at N=25,50,100,200, with persistent output/heartbeat
from threading import Thread,Event
if not START_TRAINING:
    print('Training disabled. No models were run.')
else:
    RUN_ENV=ENV.copy()
    RUN_ENV['NS_SMOKE_ONLY']='0'
    # Independent output and conservative restart: do not retrain complete results.
    RUN_ENV['NS_FORCE_RETRAIN']='1' if FORCE_RETRAIN else '0'
    logpath=OUTPUT_ROOT/'five_model_training_stdout.log'
    stopped=Event();proc_holder=[None];last=[time.monotonic()]
    def watcher():
        while not stopped.wait(60):
            proc=proc_holder[0]
            print('[HEARTBEAT] five-model process',
                  'ALIVE' if proc is not None and proc.poll() is None else 'NOT RUNNING',
                  '| last printed',round((time.monotonic()-last[0])/60,1),
                  'minutes ago |',logpath,flush=True)
    thread=Thread(target=watcher,daemon=True);thread.start()
    try:
        with logpath.open('a',encoding='utf-8') as log:
            log.write('\n=== START '+time.strftime('%Y-%m-%dT%H:%M:%S')+' ===\n')
            log.flush()
            with subprocess.Popen([sys.executable,'-u',str(DRIVER)],
                env=RUN_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                text=True,errors='replace',bufsize=1) as proc:
                proc_holder[0]=proc
                for line in proc.stdout:
                    print(line,end='',flush=True)
                    log.write(line);log.flush()
                    last[0]=time.monotonic()
                status=proc.wait()
            if status:raise RuntimeError(f'Five-model driver exit code {status}; inspect {logpath}')
    finally:
        stopped.set();thread.join(timeout=5)
    print('[SUCCESS] Requested sample-size model comparisons done.',flush=True)


In [ ]:
#@title 6. Compare original-model tables, inference times and Boussinesq-style pictures
import pandas as pd
from IPython.display import display,Image
tbl=OUTPUT_ROOT/'results'/'results_long.csv'
if not tbl.is_file():
    print('No complete sample-size set yet; training output:',OUTPUT_ROOT/'five_model_training_stdout.log')
else:
    df=pd.read_csv(tbl)
    columns=['N_train','model','protocol','full_l2_mean','full_l2_std',
             'full_l2_median','final_l2_mean','final_l2_std',
             'final_l2_median','params','real_scalar_dof',
             'training_seconds','full_test_set_inference_seconds',
             'inference_seconds_per_trajectory',
             'peak_inference_allocated_gpu_MiB']
    display(df[columns].sort_values(['N_train','model']))
    for name in ['all_models_full_l2_mean_vs_n.png',
                 'all_models_final_l2_mean_vs_n.png']:
        fig=OUTPUT_ROOT/'plots'/name
        if fig.is_file():display(Image(filename=str(fig),width=950))
    print('Additional rollout and true/predicted/error figures:',OUTPUT_ROOT/'plots')
